In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.optimize import curve_fit

In [ ]:
def run_simulation(size, tmax, interval=1, central=0, supercritical=0, height_crit=3):
    size += 2
    history = np.zeros((tmax // interval + 1, size, size), dtype=np.int8)
    avalanche = np.zeros((size, size), dtype=np.int8)

    if supercritical:
        sand = np.ones((size, size), dtype=np.int8) * 7
        sand[0, :] = 0
        sand[:, 0] = 0
        sand[-1, :] = 0
        sand[:, -1] = 0
    else:
        sand = np.zeros((size, size), dtype=np.int8)

    counter = np.zeros(tmax, dtype=np.int32)
    avalanche_freq = np.zeros(size**2, dtype=np.int32)

    sand_num = 0
    x, y = size // 2, size // 2

    for i in range(tmax):
        avalanche.fill(0)

        while sand[x, y] < 4:
            if central == 0:
                x, y = np.random.randint(1, size - 1, 2)
                
            sand[x, y] += 1
            sand_num += 1
            

        while np.any(sand > height_crit):
            ix, iy = np.where(sand > height_crit)
            avalanche[ix, iy] = 1

            sand[ix, iy] -= 4
            sand[ix+1, iy] += 1
            sand[ix-1, iy] += 1
            sand[ix, iy+1] += 1
            sand[ix, iy-1] += 1

            sand_num -= np.sum(sand[0, :] + sand[-1, :] +
                               sand[:, 0] + sand[:, -1])
            sand[0, :] = 0
            sand[:, 0] = 0
            sand[-1, :] = 0
            sand[:, -1] = 0

        cnt = np.sum(avalanche)
        avalanche_freq[cnt] += 1
        counter[i] = sand_num

        if i % interval == 0:
            history[i // interval] = sand

    return history, avalanche_freq, counter

In [ ]:
history_stack, avalanche_freq, counter = run_simulation(51, 501, central=1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')
title = ax.set_title(f"Center dropped sand (step: 0)")

img = ax.imshow(history_stack[0], cmap='hot', interpolation='nearest', vmin=0, vmax=3)

def update(frame_idx):
    current_grid = history_stack[frame_idx]
    img.set_data(current_grid)
    title.set_text(f"Center dropped sand (step: {frame_idx})")
    return [img, title]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(history_stack)-1,
    interval=50,
    blit=True
)

anim.save(f"../results/gifs/12_sandpile_avalanche_model_center.gif", writer='pillow', fps=20)
plt.close(fig)

In [ ]:
interval = 100
history_stack_top, avalanche_freq_top, counter_top = run_simulation(31, 100000, interval=interval) # ~120s / 50k

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')
title = ax.set_title(f"Random dropped sand (step: 0)")

img = ax.imshow(history_stack_top[0], cmap='hot', interpolation='nearest')

def update(frame_idx):
    current_grid = history_stack_top[frame_idx]
    img.set_data(current_grid)
    title.set_text(f"Random dropped sand (step: {frame_idx * interval})")
    return [img, title]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=min(50, len(history_stack_top)),
    interval=200,
    blit=True
)

anim.save(f"../results/gifs/12_sandpile_avalanche_model_random.gif", writer='pillow', fps=5)
plt.close(fig)

In [ ]:
def line(x, b):
  return 0 * x + b

x = np.arange(len(counter_top))
y = counter_top

fig, ax = plt.subplots()
ax.plot(x, y)
popt, pcov = curve_fit(line, x, y, p0=[2000])
ax.plot(x, line(x, *popt))
print(*popt)

ax.set_xlabel('Sand count')
ax.set_ylabel('Steps')
ax.set_title(r'Sand count over steps')
plt.show()
plt.close()


cutoff = 500
t = np.arange(cutoff)

fig, ax = plt.subplots()
ax.plot(t, counter_top[:cutoff])
ax.plot(t, line(t, *popt))

ax.set_xlabel('Sand count')
ax.set_ylabel('Steps')
ax.set_title(f'Sand count over first {cutoff} steps')
plt.show()
plt.close()

In [ ]:
def model(x, a, b):
  return a * x + b

top_cutoff = 700
x_log = np.log10(np.arange(len(avalanche_freq_top))[1:-top_cutoff])
y_log = np.log10(avalanche_freq_top[1:-top_cutoff])

fig, ax = plt.subplots()
ax.plot(x_log, y_log)

popt, pcov = curve_fit(model, x_log, y_log, p0=[-1,4])
ax.plot(x_log, model(x_log, *popt))

plt.xlabel('S [log10]')
plt.ylabel('N [log10]')
plt.title(r'Wykres zależności $N \propto S^{-1}$')
print(*popt)
plt.show()
plt.close()

In [ ]:
history_stack, avalanche_freq, counter = run_simulation(101, 500, supercritical=1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')
title = ax.set_title(f"Random dropped sand (step: 0)")

img = ax.imshow(history_stack[0], cmap='hot', interpolation='nearest')

def update(frame_idx):
    current_grid = history_stack[frame_idx]
    img.set_data(current_grid)
    title.set_text(f"Random dropped sand (step: {frame_idx})")
    return [img, title]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(history_stack),
    interval=100,
    blit=True
)

anim.save(f"../results/gifs/12_sandpile_avalanche_model_supercritical.gif", writer='pillow', fps=10)
plt.close(fig)

In [ ]:
history_stack, avalanche_freq, counter = run_simulation(101, 500, central=1, supercritical=1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')
title = ax.set_title(f"Full grid center dropped sand (step: 0)")

img = ax.imshow(history_stack[0], cmap='hot', interpolation='nearest')

def update(frame_idx):
    current_grid = history_stack[frame_idx]
    img.set_data(current_grid)
    title.set_text(f"Full grid center dropped sand (step: {frame_idx})")
    return [img, title]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(history_stack),
    interval=100,
    blit=True
)

anim.save(f"../results/gifs/12_sandpile_avalanche_model_center_supercritical.gif", writer='pillow', fps=10)
plt.close(fig)